In [1]:
import sys
sys.path.append("/mnt/d/work/smolagents/src")

In [2]:
from smolagents import FinalAnswerTool,DuckDuckGoSearchTool,Tool,tool
from smolagents._function_type_hints_utils import  get_json_schema,_convert_type_hints_to_json_schema
from typing import  Optional, Dict, Any
import os
from dotenv import load_dotenv


In [3]:
load_dotenv()

True

In [3]:
@tool
def multiply(x:float, y:float) -> float:
    """
    Multiplies two numbers.
    Args:
        x (float): The first number.
        y (float): The second number.
    Returns:
        float: The product of x and y.
    """
    return x * y

In [4]:
@tool
def add(x:float, y:float) -> float:
    """
    Adds two numbers.
    Args:
        x (float): The first number.
        y (float): The second number.
    Returns:
        float: The sum of x and y.
    """
    return x + y

# add.__class__

In [5]:
from google.genai import  types

In [8]:
schema=_convert_type_hints_to_json_schema(multiply.forward,error_on_missing_type_hints=False)

In [9]:
print(schema)

{'type': 'object', 'properties': {'x': {'type': 'number'}, 'y': {'type': 'number'}, 'return': {'type': 'number'}, 'self': {}}, 'required': ['self', 'x', 'y']}


In [10]:
_convert_type_hints_to_json_schema(DuckDuckGoSearchTool().forward,error_on_missing_type_hints=False)

{'type': 'object',
 'properties': {'query': {'type': 'string'}, 'return': {'type': 'string'}},
 'required': ['query']}

In [11]:
_convert_type_hints_to_json_schema(FinalAnswerTool().forward,error_on_missing_type_hints=False)

{'type': 'object',
 'properties': {'answer': {'type': 'any'}, 'return': {'type': 'any'}},
 'required': ['answer']}

In [12]:
multiply.inputs

{'x': {'type': 'number', 'description': 'The first number.'},
 'y': {'type': 'number', 'description': 'The second number.'}}

In [13]:
types.Schema(
    properties={
        'x': types.Schema(
            format='typing.any',
            description='The first number.'
        ),
        'y': types.Schema(
            type='number',
            description='The second number.'
        )
    },
    required=['x', 'y'],
    type='object'
)

Schema(example=None, pattern=None, default=None, max_length=None, min_length=None, min_properties=None, max_properties=None, any_of=None, description=None, enum=None, format=None, items=None, max_items=None, maximum=None, min_items=None, minimum=None, nullable=None, properties={'x': Schema(example=None, pattern=None, default=None, max_length=None, min_length=None, min_properties=None, max_properties=None, any_of=None, description='The first number.', enum=None, format='typing.any', items=None, max_items=None, maximum=None, min_items=None, minimum=None, nullable=None, properties=None, property_ordering=None, required=None, title=None, type=None), 'y': Schema(example=None, pattern=None, default=None, max_length=None, min_length=None, min_properties=None, max_properties=None, any_of=None, description='The second number.', enum=None, format=None, items=None, max_items=None, maximum=None, min_items=None, minimum=None, nullable=None, properties=None, property_ordering=None, required=None, ti

In [9]:
def get_gemini_tool(tool:Tool):
    name=tool.name
    description=tool.description
    inputs=tool.inputs
    parameters=_convert_type_hints_to_json_schema(tool.forward,error_on_missing_type_hints=False)
    required=parameters.pop('required')
    if 'self' in required:
        required.remove('self')
    inputs.pop("required",None)
    output={"name":name,"description":description,"parameters":{"type":"object","properties":inputs,"required":required}}
    return output


In [10]:
get_gemini_tool(FinalAnswerTool())

{'name': 'final_answer',
 'description': 'Provides a final answer to the given problem.',
 'parameters': {'type': 'object',
  'properties': {'answer': {'type': 'string',
    'description': 'The final answer to the problem'}},
  'required': ['answer']}}

In [11]:
types.Tool(function_declarations=[get_gemini_tool(add)])

Tool(function_declarations=[FunctionDeclaration(response=None, description='Adds two numbers.', name='add', parameters=Schema(example=None, pattern=None, default=None, max_length=None, min_length=None, min_properties=None, max_properties=None, any_of=None, description=None, enum=None, format=None, items=None, max_items=None, maximum=None, min_items=None, minimum=None, nullable=None, properties={'x': Schema(example=None, pattern=None, default=None, max_length=None, min_length=None, min_properties=None, max_properties=None, any_of=None, description='The first number.', enum=None, format=None, items=None, max_items=None, maximum=None, min_items=None, minimum=None, nullable=None, properties=None, property_ordering=None, required=None, title=None, type=<Type.NUMBER: 'NUMBER'>), 'y': Schema(example=None, pattern=None, default=None, max_length=None, min_length=None, min_properties=None, max_properties=None, any_of=None, description='The second number.', enum=None, format=None, items=None, max

In [13]:
multiply.inputs

{'x': {'type': 'number', 'description': 'The first number.'},
 'y': {'type': 'number', 'description': 'The second number.'}}

In [14]:
@tool
def some_function(x:float,y:Optional[str]=None)->str:
    """
    A function that does something.
    Args:
        x (float): The first number.
        y (Optional[str]): An optional string argument.
    Returns:
        str: A string indicating the result.
    """
    return f"Result: {x}, Optional: {y}"

In [17]:
def get_gemini_tool2(tool:Tool):
    name=tool.name
    description=tool.description
    inputs=tool.inputs
    required=[]
    for arg,desc in inputs.items():
        if 'nullable' in desc:
            desc.pop('nullable')
        else:
            required.append(arg)
    output={"name":name,"description":description,"parameters":{"type":"object","properties":inputs,"required":required}}
    return output


In [19]:
sentiment_tool = Tool.from_space(space_id="yvessadate/sentimental_analysis", name="sentiment_tool", description="A tool for sentiment analysis.",token=os.getenv("HF_TOKEN"))

Loaded as API: https://yvessadate-sentimental-analysis.hf.space ✔


Since `api_name` was not defined, it was automatically set to the first available API: `/predict`.


In [20]:
get_gemini_tool2(sentiment_tool)

{'name': 'sentiment_tool',
 'description': 'A tool for sentiment analysis.',
 'parameters': {'type': 'object',
  'properties': {'input_text': {'type': 'string', 'description': ''}},
  'required': ['input_text']}}

In [21]:
types.Tool(function_declarations=[get_gemini_tool2(sentiment_tool)])

Tool(function_declarations=[FunctionDeclaration(response=None, description='A tool for sentiment analysis.', name='sentiment_tool', parameters=Schema(example=None, pattern=None, default=None, max_length=None, min_length=None, min_properties=None, max_properties=None, any_of=None, description=None, enum=None, format=None, items=None, max_items=None, maximum=None, min_items=None, minimum=None, nullable=None, properties={'input_text': Schema(example=None, pattern=None, default=None, max_length=None, min_length=None, min_properties=None, max_properties=None, any_of=None, description='', enum=None, format=None, items=None, max_items=None, maximum=None, min_items=None, minimum=None, nullable=None, properties=None, property_ordering=None, required=None, title=None, type=<Type.STRING: 'STRING'>)}, property_ordering=None, required=['input_text'], title=None, type=<Type.OBJECT: 'OBJECT'>))], retrieval=None, google_search=None, google_search_retrieval=None, code_execution=None)

In [8]:
from smolagents import GeminiModel,ToolCallingAgent,CodeAgent

In [5]:
gemini_model = GeminiModel()

In [6]:
agent=ToolCallingAgent(model=gemini_model, tools=[DuckDuckGoSearchTool()])

In [7]:
agent.run("what is the price of latest iphone in the indian market?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ what is the price of latest iphone in the indian market?                                                        │
│                                                                                                                 │
╰─ GeminiModel - gemini-2.0-flash ────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': 'latest iPhone price in India'}                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: ## Search Results

|Apple iPhones online at Best Prices in INDIA on all Latest iPhone 
...](https://www.flipkart.com/mobiles/apple~brand/pr?sid=tyy,4io)
Buy Apple iPhone online at lowest prices. Check the latest series of iPhones : iPhone 12, iPhone 13, iPhone 14, 
iPhone 15, and iPhone 16 at Flipkart.com. Check Prices in India and Buy Online.

|iPhone - Apple (IN)](https://www.apple.com/in/iphone/)
Discover the new iPhone 16e along with iPhone 16 Pro, iPhone 16 and iPhone 15. Apple; Store; Mac; iPad; iPhone; 
Watch; ... Latest iPhone. Greatest price. * ... with Siri and device language set to Chinese (Simplified), English 
(Australia, Canada, India, Ireland, New Zealand, Singapore, South Africa, UK or US), French, German, Italian, 
Japanese ...

|Apple Mobile Phones (iPhones) Price List (Apr 2025)](https://www.91mobiles.com/apple-mobile-price-list-in-india)
Here is the list of 123 Apple mobile phones (iPhones) available in India with prices, specifications and features. 
This list was last updated on 15th April 2025. Among these, Apple iPhone 16 Pro Max, Apple iPhone 16 and Apple 
iPhone 15 stand out as the most popular phones on the list.

|Buy iPhone - Apple (IN)](https://www.apple.com/in/shop/buy-iphone)
Shop the latest iPhone models and accessories. Get up to ₹4000 instant cashback with eligible cards and get No Cost
EMI for 12 months. ... * Mac, iPad, and Apple Watch trade-in is available only in-store in India. Apple Retail 
Online in India does not offer trade-in for Mac, iPad, and Apple Watch. Trade‑in values will vary based on the ...

|Apple Mobile Phones Price List in India (Apr 2025) | Smartprix](https://www.smartprix.com/mobiles/apple-brand)
Apple Mobile Phones Price List in India; Mobile Phone Price Available From; Apple iPhone 16 ₹71,290: Sep, 2024: 
Apple iPhone 15 ₹61,390: Sep, 2023: Apple iPhone 16 Pro Max ₹1,35,900: ... This was a big week in tech—leaks, 
launches, and some serious shake-ups. We've got the latest on Nothing Phone (3a), iPhone SE 4's launch, OPPO ...

|Buy iPhone 15 and iPhone 15 Plus Unlocked - Apple (IN)](https://www.apple.com/in/shop/buy-iphone/iphone-15)
New. Latest iPhone. Greatest price. ... 15 Pro, and iPhone 15 Pro Max, with Siri and device language set to Chinese
(Simplified), English (Australia, Canada, India, Ireland, New Zealand, Singapore, South Africa, UK, or U.S.), 
French, German, Italian, Japanese, Korean, Portuguese (Brazil), or Spanish, as an iOS 18 update, with more 
languages ...

|Latest Apple Mobile Phones Price List in India - 
Smartprix](https://www.smartprix.com/mobiles/latest-apple-iphones-list)
Top 3 Latest Apple Mobile Phones Price List are as follows: Apple iPhone 16e (512GB): Wi-Fi, 5G, 6.1 inches, Hexa 
Core, 4005 mAh Battery with Fast Charging Apple iPhone 16e (256GB): 1170 x 2532 px Display with Small Notch, iOS 
v18, 8 GB RAM, 3G, Wi-Fi Apple iPhone 16e: Dual Sim, 6.1 inches, 48 MP Rear & 12 MP Front Camera, 3G, 4.04 GHz 
Processor

|Apple iPhone: Buy iPhone Online at Best Price | 
Croma](https://www.croma.com/phones-wearables/mobile-phones/iphones/c/97)
Explore our vast collection now and browse Apple iPhone prices to find the one that fits your range. Why Buy 
iPhone- Top Features, Specs and Price ... Get great deals on iPhone latest models like iPhone 15, iPhone 15Pro& 
more only at Croma. Check out iPhones Phone Price, reviews, features and more. ... Buy Apple iPhone Cell Phones 
Online in India.

|Apple Mobile Phones Price List (Apr 2025) - 
MySmartPrice](https://www.mysmartprice.com/mobile/pricelist/apple-mobile-price-list-in-india.html)
Sure, the iPhone 15 may seem like it costs a pretty penny, but it brings a lot of important Apple tech such as USB 
Type-C, 48MP camera, Dynamic Island and the A16 Bionic chip. Despite the lack of a telephoto camera on the iPhone 
15, its computational photography prowess helps it be one of the best camera phones for the price point.

|Buy iPhone Online - Buy Latest iPhones, Apple Phones Online - Vijay S

[Step 1: Duration 2.52 seconds]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The price of the latest iPhone in the Indian market    │
│ starts from approximately ₹61,390 (iPhone 15) and can go up to ₹1,35,900 for the expected iPhone 16 Pro Max.    │
│ These prices may vary depending on the retailer and storage capacity.'}                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Final answer: The price of the latest iPhone in the Indian market starts from approximately ₹61,390 (iPhone 15) and
can go up to ₹1,35,900 for the expected iPhone 16 Pro Max. These prices may vary depending on the retailer and 
storage capacity.

[Step 2: Duration 1.04 seconds]

'The price of the latest iPhone in the Indian market starts from approximately ₹61,390 (iPhone 15) and can go up to ₹1,35,900 for the expected iPhone 16 Pro Max. These prices may vary depending on the retailer and storage capacity.'

In [9]:
agent=CodeAgent(model=gemini_model, tools=[DuckDuckGoSearchTool()])

In [12]:
agent.run("what is the price of latest iphone in the indian market?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ what is the price of latest iphone in the indian market?                                                        │
│                                                                                                                 │
╰─ GeminiModel - gemini-2.0-flash ────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Request contains an invalid argument.', 'status': 
'INVALID_ARGUMENT'}}

[Step 1: Duration 308.91 seconds]

AgentGenerationError: Error in generating model output:
400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Request contains an invalid argument.', 'status': 'INVALID_ARGUMENT'}}